In [1]:
### Dependencies
import sys
from types import SimpleNamespace
from datetime import datetime, timedelta, timezone
import time

import boto3
from botocore.exceptions import ClientError

In [2]:
### config
def load_config() -> SimpleNamespace:
    # TODO: now we hardcoded porperties
    config = SimpleNamespace(
        # MinIO connections
        minio_endpoint_url="http://localhost:9000",
        minio_region="us-east-1",
        # MinIO credentials
        access_key_id="minioadmin",
        secret_access_key="minioadmin",
        # bucket-related
        bucket="simres", # bucket name must be lowercase
        # object related
        file="img.png",
        retention_days=7
    )

    return config

In [3]:
### All supporting functions
## S3/MinIO client
def create_s3_client(args: SimpleNamespace):
    # explicit session
    session_kwargs = {}
    session_kwargs["aws_access_key_id"] = args.access_key_id
    session_kwargs["aws_secret_access_key"] = args.secret_access_key

    session = boto3.Session(**session_kwargs)

    return session.client(
        "s3",
        endpoint_url=args.minio_endpoint_url,
        region_name=args.minio_region
    )

## bucket-related operations
def bucket_exists(s3_client, bucket: str) -> bool:
    try:
        s3_client.head_bucket(Bucket=bucket)
        return True
    except ClientError as e:
        # https://docs.aws.amazon.com/boto3/latest/reference/services/s3/client/head_bucket.html
        # If the bucket doesn’t exist or you don’t have permission to access it, the HEAD request returns a generic 400 Bad Request, 403 Forbidden, or 404 Not Found HTTP status code. A message body isn’t included, so you can’t determine the exception beyond these HTTP response codes.
        return False

def create_bucket(s3_client, bucket: str):
    if bucket_exists(s3_client, bucket):
        print(f"Bucket {bucket} already exists")
        return
    
    s3_client.create_bucket(
        Bucket=bucket,
        ObjectLockEnabledForBucket=True # Retention: Bucket must enable Object Lock FIRST
    )
    print(f"Bucket {bucket} created")

def list_buckets(s3_client):
    response = s3_client.list_buckets()
    buckets = response.get("Buckets", [])
    
    if not buckets:
        print("No buckets found")
    else:
        print("Buckets:")
        for bucket in buckets:
            print(f"  - {bucket['Name']}")

def delete_bucket(s3_client, bucket: str):
    if not bucket_exists(s3_client, bucket):
        print(f"Bucket {bucket} does not exist")
        return
    
    # verify object versions + markers
    response = s3_client.list_object_versions(Bucket=bucket)

    versions = response.get("Versions", [])
    delete_markers = response.get("DeleteMarkers", [])

    if versions or delete_markers:
        print(
            f"Bucket {bucket} still contains "
            f"object versions or delete markers"
        )
        return

    s3_client.delete_bucket(Bucket=bucket)
    print(f"Bucket {bucket} deleted")

## object-related operations
def object_exists(s3_client, bucket: str, key: str) -> bool:
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError as e:
        return False

def upload_object(s3_client,
                  bucket: str,
                  key: str,
                  file_path: str,
                  retention: bool,
                  retention_days: int = 0):
    s3_client.upload_file(file_path, bucket, key)
    print(f"Object {key} uploaded to bucket {bucket}")

    if retention:
        s3_client.put_object_retention(
            Bucket=bucket,
            Key=key,
            Retention={
                "Mode": "GOVERNANCE",
                "RetainUntilDate": (datetime.now(timezone.utc) + timedelta(days=retention_days)) # S3/MinIO retention should use timezone-aware UTC timestamps
            }
        )
        print(f"Retention set for object {key} in bucket {bucket}")

def list_objects(s3_client, bucket: str):
    response = s3_client.list_objects_v2(Bucket=bucket)
    objects = response.get("Contents", [])
    
    if not objects:
        print(f"No objects found in bucket {bucket}")
    else:
        print(f"Objects in bucket {bucket}:")
        for obj in objects:
            print(f"  - {obj['Key']}")

def get_object_metadata(s3_client, bucket: str, key: str):
    try:
        response = s3_client.head_object(Bucket=bucket, Key=key)
        print(f"Metadata for object {key} in bucket {bucket}:")
        for k, v in response.items():
            print(f"  - {k}: {v}")
    except ClientError as e:
        # TODO: may be other reasons for failure, not just object not exist
        # TODO: or use object_exists() before calling head_object()
        print(f"Object {key} does not exist in bucket {bucket}")

def download_object(s3_client, bucket: str, key: str, download_path: str):
    if not object_exists(s3_client, bucket, key):
        print(f"Object {key} does not exist in bucket {bucket}")
        return
    
    s3_client.download_file(bucket, key, download_path)
    print(f"Object {key} downloaded to {download_path}")

def delete_object(s3_client, bucket: str, key: str):
    if not object_exists(s3_client, bucket, key):
        print(f"Object {key} does not exist in bucket {bucket}")
        return
    try:
        s3_client.delete_object(Bucket=bucket, Key=key, BypassGovernanceRetention=False)
        print(f"Object {key} deleted from bucket {bucket}")
    except ClientError as e:
        print(f"Error occurred while deleting object {key} from bucket {bucket}: {e}")

def fully_delete_object(s3_client, bucket: str, key: str):
    response = s3_client.list_object_versions(
        Bucket=bucket,
        Prefix=key
    )

    versions = response.get("Versions", [])
    for version in versions:
        if version["Key"] == key:
            try:
                s3_client.delete_object(
                    Bucket=bucket,
                    Key=key,
                    VersionId=version["VersionId"]
                )
                print(f"Deleted version: {version['VersionId']}")
            except ClientError as e:
                print(f"Error occurred while deleting version {version['VersionId']} of object {key} from bucket {bucket}: {e}")

    delete_markers = response.get("DeleteMarkers", [])
    for marker in delete_markers:
        if marker["Key"] == key:
            s3_client.delete_object(
                Bucket=bucket,
                Key=key,
                VersionId=marker["VersionId"]
            )
            print(f"Deleted delete marker: {marker['VersionId']}")

# TODO: instead, we can set bypass governance retention when deleting the object, which will work even if the object is under retention
def remove_object_retention(s3_client, bucket: str, key: str):
    response = s3_client.list_object_versions(
        Bucket=bucket,
        Prefix=key
    )

    versions = response.get("Versions", [])

    for version in versions:
        if version["Key"] == key:
            s3_client.put_object_retention(
                Bucket=bucket,
                Key=key,
                VersionId=version["VersionId"], # IMPORTANT!!
                Retention={
                    "Mode": "GOVERNANCE",
                    "RetainUntilDate": (datetime.now(timezone.utc) + timedelta(seconds=1))
                },
                BypassGovernanceRetention=True
            )

            print(f"Retention removed for object {key}, version {version['VersionId']} in bucket {bucket}")

    time.sleep(1)

def recover_deleted_object(s3_client, bucket: str, key: str):
    response = s3_client.list_object_versions(
        Bucket=bucket,
        Prefix=key
    )

    delete_markers = response.get("DeleteMarkers", [])

    latest_marker = None

    for marker in delete_markers:
        if marker["Key"] == key and marker.get("IsLatest"):
            latest_marker = marker
            break

    if latest_marker is None:
        print(f"No latest delete marker found for {key}")
        return

    s3_client.delete_object(
        Bucket=bucket,
        Key=key,
        VersionId=latest_marker["VersionId"]
    )

    print(
        f"Recovered {key} by deleting delete marker "
        f"{latest_marker['VersionId']}"
    )

In [4]:
### Load config and connect to MinIO
args = load_config()

s3_client = create_s3_client(args)

In [5]:
### 1. List all buckets
## Expect: no bucket at the beginning
list_buckets(s3_client)

No buckets found


In [6]:
### 2. Create a bucket
create_bucket(s3_client, args.bucket)

Bucket simres created


In [7]:
### 3. List all buckets and objects
## Expect: an empty bucket
list_buckets(s3_client)

list_objects(s3_client, args.bucket)

Buckets:
  - simres
No objects found in bucket simres


In [8]:
### 4. Upload two files, one without retention and one with retention
# without retention
upload_object(s3_client, args.bucket, "OBJECT_1", args.file, False)

# with retention
upload_object(s3_client, args.bucket, "OBJECT_2", args.file, True, args.retention_days)

list_objects(s3_client, args.bucket)

Object OBJECT_1 uploaded to bucket simres
Object OBJECT_2 uploaded to bucket simres
Retention set for object OBJECT_2 in bucket simres
Objects in bucket simres:
  - OBJECT_1
  - OBJECT_2


In [9]:
### 5. Get metadata
get_object_metadata(s3_client, args.bucket, "OBJECT_1")

Metadata for object OBJECT_1 in bucket simres:
  - ResponseMetadata: {'RequestId': '18B16C9F62157518', 'HostId': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'HTTPStatusCode': 200, 'HTTPHeaders': {'accept-ranges': 'bytes', 'content-length': '60290', 'content-type': 'binary/octet-stream', 'etag': '"5ce9f45d34ec47cf0c9801da2b534da3"', 'last-modified': 'Thu, 21 May 2026 00:19:19 GMT', 'server': 'MinIO', 'strict-transport-security': 'max-age=31536000; includeSubDomains', 'vary': 'Origin, Accept-Encoding', 'x-amz-id-2': 'dd9025bab4ad464b049177c95eb6ebf374d3b3fd1af9251148b658df7ac2e3e8', 'x-amz-request-id': '18B16C9F62157518', 'x-content-type-options': 'nosniff', 'x-ratelimit-limit': '2060', 'x-ratelimit-remaining': '2060', 'x-xss-protection': '1; mode=block', 'x-amz-version-id': '63de7e9c-e443-4a62-82ea-8b0e27249b5e', 'date': 'Thu, 21 May 2026 00:19:19 GMT'}, 'RetryAttempts': 0}
  - AcceptRanges: bytes
  - LastModified: 2026-05-21 00:19:19+00:00
  - ContentLength: 602

In [10]:
### 6. Download object
download_object(s3_client, args.bucket, "OBJECT_1", datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3] + ".png")

Object OBJECT_1 downloaded to 20260520_171919_598.png


In [11]:
### 7. Soft delete objects
## Expect: no listed objects, but versions and markers still exist
delete_object(s3_client, args.bucket, "OBJECT_1")

delete_object(s3_client, args.bucket, "OBJECT_2")

list_objects(s3_client, args.bucket)

Object OBJECT_1 deleted from bucket simres
Object OBJECT_2 deleted from bucket simres
No objects found in bucket simres


In [12]:
### 8. Recover deleted objects
## Expect: OBJECT_1 listed
recover_deleted_object(s3_client, args.bucket, "OBJECT_1")

list_objects(s3_client, args.bucket)

Recovered OBJECT_1 by deleting delete marker 6fc85c95-151e-40fd-abb4-672c402c537c
Objects in bucket simres:
  - OBJECT_1


In [13]:
### 9. Fully deleted objects with retention set
## Expect: OBJECT_2 cannot fully deleted, so bucket cannot be deleted
fully_delete_object(s3_client, args.bucket, "OBJECT_1")

fully_delete_object(s3_client, args.bucket, "OBJECT_2")

delete_bucket(s3_client, args.bucket)

Deleted version: 63de7e9c-e443-4a62-82ea-8b0e27249b5e
Error occurred while deleting version 9000d875-4081-471e-905f-45acba4ae883 of object OBJECT_2 from bucket simres: An error occurred (InvalidRequest) when calling the DeleteObject operation: Object is WORM protected and cannot be overwritten
Deleted delete marker: 2c42ee02-f351-4674-bec6-46362044121d
Bucket simres still contains object versions or delete markers


In [14]:
### 10. Remove retention
## Expect: OBJECT_2 can be fully removed
remove_object_retention(s3_client, args.bucket, "OBJECT_2")

fully_delete_object(s3_client, args.bucket, "OBJECT_2")

Retention removed for object OBJECT_2, version 9000d875-4081-471e-905f-45acba4ae883 in bucket simres
Deleted version: 9000d875-4081-471e-905f-45acba4ae883


In [15]:
### 11. Remove bucket as clean up
## Expect: no bucket listed
delete_bucket(s3_client, args.bucket)
    
list_buckets(s3_client)

Bucket simres deleted
No buckets found
